# DCC - Dynamic Conditional Correlation (Solution)

**Referencia**: Engle, R. (2002). *Dynamic Conditional Correlation: A Simple Class of Multivariate Generalized Autoregressive Conditional Heteroskedasticity Models*. Journal of Business & Economic Statistics, 20(3), 339-350.

---

O modelo DCC estende o CCC permitindo que as **correlacoes variem ao longo do tempo**.
Isso e fundamental para capturar fenomenos como **contagio financeiro** e **mudancas de regime**.

### Estrutura do modelo

$$H_t = D_t \, R_t \, D_t$$

onde $R_t$ agora e **dinamica**, seguindo:

$$Q_t = (1 - a - b) \bar{Q} + a \, z_{t-1} z_{t-1}' + b \, Q_{t-1}$$
$$R_t = \text{diag}(Q_t)^{-1/2} \, Q_t \, \text{diag}(Q_t)^{-1/2}$$

### Neste notebook

1. Evidencia de correlacoes variando no tempo
2. Estimacao do modelo DCC
3. Correlacoes condicionais dinamicas
4. Comparacao CCC vs DCC
5. Previsao de correlacoes
6. Aplicacao: otimizacao de portfolio

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Adicionar utils ao path
sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import (
    plot_correlation_heatmap,
    plot_dynamic_correlations,
    plot_portfolio_weights,
)

from archbox.multivariate import CCC, DCC

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

# Carregar dados fx_majors (4 series)
data_path = os.path.join("..", "data", "fx_majors.csv")
df = pd.read_csv(data_path, parse_dates=["date"], index_col="date")
returns = df.values
labels = [col.upper() for col in df.columns]
dates = df.index

print(f"Dataset: {df.shape[0]} obs x {df.shape[1]} series")
print(f"Series: {labels}")

## 1. Por que correlacoes dinamicas?

Em periodos de **crise financeira**, as correlacoes entre ativos tendem a aumentar
drasticamente - um fenomeno conhecido como **contagio**. Isso viola a hipotese de
correlacao constante do CCC.

Evidencias empiricas mostram que:
- Correlacoes entre moedas aumentam em periodos de aversao ao risco
- O fenomeno "flight to quality" altera as co-movimentacoes
- Correlacoes sao **persistentes** mas nao constantes

In [ ]:
# Mostre evidencia de correlacoes variando no tempo (rolling corr)

window = 250  # janela de 1 ano

# Calcular correlacoes rolling para todos os pares
rolling_corrs = {}
k = len(labels)
for i in range(k):
    for j in range(i+1, k):
        pair_name = f"{labels[i]}-{labels[j]}"
        rolling_corrs[pair_name] = df.iloc[:, i].rolling(window).corr(df.iloc[:, j]).values

# Plotar evidencia
plot_dynamic_correlations(
    dates, rolling_corrs,
    title=f"Correlacoes Rolling ({window} dias) - Evidencia de Variacao Temporal"
)
plt.show()

# Estatisticas das correlacoes rolling
print("Variabilidade das correlacoes rolling:")
for pair, corr_series in rolling_corrs.items():
    valid = corr_series[~np.isnan(corr_series)]
    print(f"  {pair}: media={np.mean(valid):.3f}, std={np.std(valid):.3f}, "
          f"min={np.min(valid):.3f}, max={np.max(valid):.3f}")

## 2. O modelo DCC

O DCC estima correlacoes dinamicas em duas etapas:

**Etapa 1** - Volatilidades univariadas (identica ao CCC):
$$\sigma_{i,t}^2 = \omega_i + \alpha_i \varepsilon_{i,t-1}^2 + \beta_i \sigma_{i,t-1}^2$$

**Etapa 2** - Dinamica de correlacoes:
$$Q_t = (1 - a - b) \bar{Q} + a \, z_{t-1} z_{t-1}' + b \, Q_{t-1}$$

onde:
- $\bar{Q} = \frac{1}{T} \sum_{t=1}^T z_t z_t'$ e a correlacao incondicional dos residuos padronizados
- $a$ captura o impacto de **choques recentes** nas correlacoes
- $b$ controla a **persistencia** das correlacoes
- $a + b < 1$ garante estacionariedade

A correlacao condicional e normalizada:
$$R_t = \text{diag}(Q_t)^{-1/2} \, Q_t \, \text{diag}(Q_t)^{-1/2}$$

In [ ]:
# Estime DCC com archbox

# Estimar modelo DCC-GARCH(1,1)
model_dcc = DCC(returns, univariate_model="GARCH", univariate_order=(1, 1))
results_dcc = model_dcc.fit(method="two_step", disp=True)

# Resultados
print(f"\nLog-likelihood: {results_dcc.loglike:.4f}")
print(f"AIC: {results_dcc.aic:.4f}")
print(f"BIC: {results_dcc.bic:.4f}")

# Parametros DCC
print(f"\nParametros DCC (a, b): {results_dcc.params}")
if len(results_dcc.params) >= 2:
    a, b = results_dcc.params[0], results_dcc.params[1]
    print(f"  a (news impact): {a:.6f}")
    print(f"  b (persistence): {b:.6f}")
    print(f"  a + b (total persistence): {a + b:.6f}")

## 3. Correlacoes condicionais dinamicas

Com 4 series, temos $\binom{4}{2} = 6$ pares de correlacoes condicionais.
Estas correlacoes variam ao longo do tempo, capturando mudancas na
co-movimentacao entre as moedas.

In [ ]:
# Plote as 6 correlacoes condicionais (4 choose 2 pares)

R_t = results_dcc.dynamic_correlation  # shape: (T, k, k)

# Extrair os 6 pares de correlacoes
dcc_corrs = {}
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        pair_name = f"{labels[i]}-{labels[j]}"
        dcc_corrs[pair_name] = R_t[:, i, j]

# Plotar correlacoes dinamicas
plot_dynamic_correlations(
    dates, dcc_corrs,
    title="DCC - Correlacoes Condicionais Dinamicas"
)
plt.show()

# Heatmap da correlacao media
R_mean = np.mean(R_t, axis=0)
plot_correlation_heatmap(R_mean, labels, title="Correlacao Condicional Media (DCC)")
plt.show()

## 4. Comparacao CCC vs DCC

Podemos comparar os modelos usando criterios de informacao:
- **AIC** (Akaike): $-2 \ln L + 2k$
- **BIC** (Bayesian): $-2 \ln L + k \ln T$

O teste de **Engle-Sheppard** testa formalmente $H_0$: correlacoes constantes vs.
$H_1$: correlacoes dinamicas.

In [ ]:
# Compare AIC/BIC de CCC vs DCC

# Estimar CCC para comparacao
model_ccc = CCC(returns, univariate_model="GARCH", univariate_order=(1, 1))
results_ccc = model_ccc.fit(method="two_step", disp=False)

# Tabela comparativa
comparison = pd.DataFrame({
    "Modelo": ["CCC", "DCC"],
    "Log-Likelihood": [results_ccc.loglike, results_dcc.loglike],
    "AIC": [results_ccc.aic, results_dcc.aic],
    "BIC": [results_ccc.bic, results_dcc.bic],
    "N Params (corr)": [len(results_ccc.params), len(results_dcc.params)],
})
comparison = comparison.set_index("Modelo")
print("Comparacao CCC vs DCC:")
print(comparison.to_string())

# Identificar melhor modelo
best_aic = "CCC" if results_ccc.aic < results_dcc.aic else "DCC"
best_bic = "CCC" if results_ccc.bic < results_dcc.bic else "DCC"
print(f"\nMelhor pelo AIC: {best_aic}")
print(f"Melhor pelo BIC: {best_bic}")

## 5. Previsao de correlacoes

O DCC permite prever a estrutura de correlacao futura. Para $h$ passos a frente:

$$E[Q_{t+h} | \mathcal{F}_t] = \bar{Q} + (a + b)^h (Q_t - \bar{Q})$$

As correlacoes previstas **convergem para $\bar{Q}$** a medida que o horizonte aumenta,
com velocidade determinada por $a + b$.

In [ ]:
# Faca previsao de correlacoes para 10 dias

# Previsao
horizon = 10
forecast = model_dcc.forecast(results_dcc, horizon=horizon)

print(f"Previsao de correlacoes para {horizon} dias:")
print(f"Chaves do forecast: {list(forecast.keys())}")

# Visualizar convergencia das correlacoes previstas
if "correlation" in forecast:
    R_forecast = forecast["correlation"]  # shape: (horizon, k, k)

    fig, ax = plt.subplots(figsize=(10, 6))
    horizons = np.arange(1, horizon + 1)

    for i in range(len(labels)):
        for j in range(i+1, len(labels)):
            pair_name = f"{labels[i]}-{labels[j]}"
            ax.plot(horizons, R_forecast[:, i, j], "o-", label=pair_name, markersize=4)

    ax.set_title("Previsao de Correlacoes Condicionais (DCC)")
    ax.set_xlabel("Horizonte (dias)")
    ax.set_ylabel("Correlacao Prevista")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
elif "covariance" in forecast:
    H_forecast = forecast["covariance"]  # shape: (horizon, k, k)
    print("\nMatriz de covariancia prevista (h=1):")
    print(pd.DataFrame(H_forecast[0], index=labels, columns=labels).round(6))
    print("\nMatriz de covariancia prevista (h=10):")
    print(pd.DataFrame(H_forecast[-1], index=labels, columns=labels).round(6))

## 6. Aplicacao: otimizacao de portfolio

Uma aplicacao direta do DCC e a **otimizacao de portfolio de minima variancia**.
Usando a matriz de covariancia condicional $H_t$, os pesos otimos sao:

$$w_t = \frac{H_t^{-1} \mathbf{1}}{\mathbf{1}' H_t^{-1} \mathbf{1}}$$

Os pesos mudam ao longo do tempo conforme as covariancias condicionais se ajustam.

In [ ]:
# Use H_t do DCC para calcular portfolio de minima variancia

H_t = results_dcc.dynamic_covariance  # shape: (T, k, k)
T = H_t.shape[0]
k = H_t.shape[1]
ones = np.ones(k)

# Calcular pesos de minima variancia para cada t
weights = np.zeros((T, k))
for t in range(T):
    try:
        H_inv = np.linalg.inv(H_t[t])
        w = H_inv @ ones / (ones @ H_inv @ ones)
        weights[t] = w
    except np.linalg.LinAlgError:
        weights[t] = weights[t-1] if t > 0 else np.ones(k) / k

# Plotar pesos ao longo do tempo
plot_portfolio_weights(
    dates, weights, labels,
    title="Portfolio de Minima Variancia (DCC)"
)
plt.show()

# Estatisticas dos pesos
print("Estatisticas dos pesos do portfolio:")
weights_df = pd.DataFrame(weights, index=dates, columns=labels)
print(weights_df.describe().round(4))

# Retorno do portfolio
portfolio_returns = np.sum(returns * weights, axis=1)
equal_weight_returns = np.mean(returns, axis=1)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(dates, np.cumsum(portfolio_returns), label="Min-Variance (DCC)", linewidth=1.2)
ax.plot(dates, np.cumsum(equal_weight_returns), label="Equal Weight", linewidth=1.2, alpha=0.7)
ax.set_title("Retorno Acumulado: Min-Variance vs Equal Weight")
ax.set_xlabel("Data")
ax.set_ylabel("Retorno Acumulado")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"\nVolatilidade anualizada (Min-Var): {np.std(portfolio_returns) * np.sqrt(252):.4f}")
print(f"Volatilidade anualizada (Equal-W): {np.std(equal_weight_returns) * np.sqrt(252):.4f}")